# Atelier 24 · LoRA sur un petit classifieur d'alertes

Carnet **facultatif**, à exécuter sur un GPU gratuit en ligne (par exemple Google Colab, environnement « T4 GPU »).

- Données : alertes **fictives** générées par `ateliers/24-lora/make_dataset.py`.
- Modèle : `distilbert-base-uncased`, affiné avec **LoRA** (poids d'origine gelés, petites matrices entraînées).
- Limite assumée : le jeu de test reformule les mêmes alertes que l'entraînement ; il illustre la méthode, il ne mesure pas la généralisation à des incidents inconnus.

Ce carnet télécharge des paquets et un modèle depuis Internet : ne l'exécutez pas sur un poste qui doit rester hors ligne.

In [ ]:
# Versions épinglées pour ce carnet (pas des dépendances du projet)
%pip install -q transformers==4.46.3 peft==0.13.2 datasets==3.1.0 accelerate==1.1.1

In [ ]:
# Récupérer le projet pour utiliser le générateur de données
!git clone --depth 1 https://github.com/arabaaoui/ops-assist.git
import sys

sys.path.insert(0, "ops-assist/ateliers/24-lora")
from make_dataset import make_dataset

data = make_dataset()
labels = sorted({row["label"] for row in data["train"] + data["test"]})
label2id = {label: i for i, label in enumerate(labels)}
print(
    len(data["train"]), "exemples d'entraînement,", len(data["test"]), "de test, classes :", labels
)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)


def encode(rows):
    """Tokenize labelled rows into a Hugging Face dataset."""
    ds = Dataset.from_list([{"text": r["text"], "label": label2id[r["label"]]} for r in rows])
    return ds.map(
        lambda b: tokenizer(b["text"], truncation=True, padding="max_length", max_length=64),
        batched=True,
    )


train_ds, test_ds = encode(data["train"]), encode(data["test"])

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForSequenceClassification

base = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(labels))
config = LoraConfig(
    task_type="SEQ_CLS", r=8, lora_alpha=16, lora_dropout=0.1, target_modules=["q_lin", "v_lin"]
)
model = get_peft_model(base, config)
model.print_trainable_parameters()  # observez la part de paramètres entraînables

In [ ]:
import numpy as np
from transformers import Trainer, TrainingArguments


def accuracy(pred):
    """Share of correctly classified test alerts."""
    return {"accuracy": float((np.argmax(pred.predictions, axis=-1) == pred.label_ids).mean())}


args = TrainingArguments(
    output_dir="lora-alerts",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    learning_rate=5e-4,
    logging_steps=5,
    report_to=[],
    seed=42,
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=accuracy
)
trainer.train()
print(trainer.evaluate())

## Comparaison avec un modèle généraliste, sans entraînement

Cellule facultative : elle appelle un modèle hébergé via LiteLLM, avec **votre** clé (ici `GEMINI_API_KEY`, ou un autre fournisseur). Ne mettez jamais la clé dans le carnet : utilisez les secrets de l'environnement.

In [ ]:
%pip install -q litellm==1.109.1
import os

import litellm

MODEL_ID = os.environ.get("ZERO_SHOT_MODEL", "gemini/gemini-2.5-flash-lite")
correct = 0
for row in data["test"]:
    prompt = (
        f"Classify this alert into exactly one of {labels}. "
        f"Answer with the label only.\n\nAlert: {row['text']}"
    )
    answer = litellm.completion(model=MODEL_ID, messages=[{"role": "user", "content": prompt}])
    correct += answer.choices[0].message.content.strip() == row["label"]
print(f"zéro-shot : {correct}/{len(data['test'])}")

## À retenir

- Comparez les deux scores **sur le même jeu de test**, et notez-les dans l'atelier.
- Le petit modèle affiné est rapide et peu coûteux, mais il ne connaît que ces classes.
- Vérifiez la licence du modèle de base avant toute réutilisation.
- Un adaptateur entraîné pour une version du modèle de base doit être réentraîné pour la suivante.